# v11 — MLA latent-KV decode — the gate (T4 / Colab; the SHAPE change)

v11 forks v10 (NVFP4 KV) changing **one variable — the attention SHAPE**: GQA-over-`H_kv`-heads -> **MQA-over-ONE-shared-latent**. All `h_q` query heads share the single latent (M = `h_q`, not M = G), so >1 warp is active at N_q=1 — the per-CTA wall v10 proved is the decode limiter — and decode AI rises `2G/b` -> `~3.78*h_q/b`.

**This Colab gate covers correctness + capacity + accuracy (Gate 1).** The kernel is CUDA-core / dequant-to-FP16 with NVFP4 latent storage carried byte-identical from v10. **T4 latency is NOT valid** (emulated FP4 unpack is software ALU; M=128 tensor-core packing is the B300 story) — read the A/B for the byte-isolation SHAPE and the **%HBM trend** (does MLA leave the per-CTA floor?), NOT absolute us/tok. The latency/regime/crossover record + the native-FP4 arm are the `[RENT root B300/sm_103]` deliverables (cells 9-10).


## 0. Dependencies + GPU (venv-safe)

In [1]:
import os, sys, subprocess

def pip(*pkgs, extra=()):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs, *extra], check=True)

# 1) Physical GPU on this runtime? (Colab defaults to CPU; pick a GPU explicitly.)
try:
    has_gpu = subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
except FileNotFoundError:
    has_gpu = False
if not has_gpu:
    raise SystemExit(
        'No GPU on this Colab runtime. FIX: Runtime > Change runtime type > T4 GPU > Save, '
        'then Runtime > Restart session, then re-run from the top. (This kernel needs a Turing T4.)')

# 2) Install deps (incl. numpy) BEFORE importing torch, so torch's numpy bridge initializes.
pip('ninja', 'pytest', 'numpy')

# 3) torch present AND CUDA-enabled? A CPU-only wheel raises "not compiled with CUDA" on any kernel.
try:
    import torch
    cuda_ok = torch.cuda.is_available()
except ImportError:
    torch, cuda_ok = None, False

if not cuda_ok:
    pip('torch', extra=('--index-url', 'https://download.pytorch.org/whl/cu124'))
    raise SystemExit(
        'A GPU is present but torch was a CPU-only build -- installed the CUDA build. NOW: '
        'restart the kernel/session, then re-run this cell.')

# vast.ai/venv: !-cells spawn a bare shell without the venv on PATH -> `python` not found.
os.environ['PATH'] = os.path.dirname(sys.executable) + os.pathsep + os.environ.get('PATH', '')

# torch.float8_e4m3fn must exist (>=2.1) — the NVFP4 latent's per-16 micro-scale is an E4M3 byte.
assert hasattr(torch, 'float8_e4m3fn'), 'this torch lacks float8_e4m3fn; upgrade torch (>=2.1)'
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| cap', torch.cuda.get_device_capability())
!nvidia-smi --query-gpu=name,compute_cap --format=csv

torch 2.11.0+cu128 | cuda 12.8 | cap (7, 5)
name, compute_cap
Tesla T4, 7.5


## 1. Get the repo

In [2]:
REPO_URL = 'https://github.com/gkienpham-cmd/flashattention-cuda.git'  # public; plain clone works
import os, sys, subprocess
if os.path.basename(os.getcwd()) != 'flashattention-cuda':
    if not os.path.isdir('flashattention-cuda'):
        subprocess.run(['git', 'clone', REPO_URL], check=True)
    os.chdir('flashattention-cuda')
subprocess.run(['git', 'pull', 'origin', 'main'])
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print('cwd', os.getcwd())

cwd /content/flashattention-cuda


## 2. Roofline — MLA lifts decode AI ~30x (recorded BEFORE coding; results.md Step 11)

MLA shares ONE latent across all `h_q` heads -> decode AI = `2*h_q*(2L+R)/((L+R)*b)` ~= `3.78*h_q/b` (L=512, R=64). At h_q=128, fp16 that is ~235 (vs GQA-8's 8) — near the B300 FP16-TC ridge (312). The **pure roofline** says fp8/nvfp4 latent FLIP it compute-bound; the **per-CTA-corrected** real prediction is a RENAME to smem-capacity (staging the latent caps occupancy ~1 block/SM).

In [3]:
from roofline.archs import get_arch
from roofline.model import estimate
# MLA decode AI for the h_q M-packing sweep, vs the GQA-8 baseline. Run box is the T4 (sm_75); the
# paper's prediction is on B300 (sm_103). b: fp16=2, fp8=1, nvfp4=0.5625.
L, R = 512, 64
for sm in ('sm_75', 'sm_103'):
    arch = get_arch(sm)
    ridge = arch.fp16_tc_flops / (arch.hbm_bw_gbps * 1e9)
    g8 = estimate(arch, B=1, H=8, N_q=1, N_k=8192, d=128, precision='fp16', G=8)  # GQA-8 fp16 baseline
    print(f'\n=== {arch.name} ({sm}) | HBM {arch.hbm_bw_gbps} GB/s | FP16-TC ridge {ridge:.1f} | GQA-8 fp16 AI {g8.arithmetic_intensity:.1f} ===')
    print(f"{'h_q':>4} | {'AI fp16':>8} | {'AI fp8':>8} | {'AI nvfp4':>9} | {'lim(fp16)':>9} | {'lim(nvfp4)':>10}")
    for h_q in (16, 32, 64, 128):
        e16 = estimate(arch, B=1, H=1, N_q=1, N_k=8192, d=L+R, precision='fp16',  mla=True, h_q=h_q, kv_lora_rank=L, rope_dim=R)
        e8  = estimate(arch, B=1, H=1, N_q=1, N_k=8192, d=L+R, precision='fp8',   mla=True, h_q=h_q, kv_lora_rank=L, rope_dim=R)
        e4  = estimate(arch, B=1, H=1, N_q=1, N_k=8192, d=L+R, precision='nvfp4', mla=True, h_q=h_q, kv_lora_rank=L, rope_dim=R)
        print(f'{h_q:>4} | {e16.arithmetic_intensity:8.1f} | {e8.arithmetic_intensity:8.1f} | '
              f'{e4.arithmetic_intensity:9.1f} | {e16.limiter.upper():>9} | {e4.limiter.upper():>10}')
print('\nPURE ROOFLINE: at h_q=128, fp16 sits ~0.75x the FP16-TC ridge; fp8/nvfp4 latent push AI PAST it')
print('-> a limiter FLIP to compute (first in the v1->v11 arc). PER-CTA-CORRECTED (the real prediction):')
print('the model is BLIND to on-chip capacity -> staging the latent (~46 KB at the real 576/512 shape,')
print('vs the T4 48 KB static limit) likely caps occupancy at ~1 block/SM and RENAMES the limiter to')
print('smem-capacity. COUNTER (the prize): if measured %HBM stays at the ~0.5%/~40 GB/s per-CTA floor,')
print('MLA did NOT leave per-CTA-bound -> the speculative q_len>1 shape-lever is the fallback. Either sign publishable.')


=== Tesla T4 (sm_75) | HBM 320.0 GB/s | FP16-TC ridge 203.1 | GQA-8 fp16 AI 8.0 ===
 h_q |  AI fp16 |   AI fp8 |  AI nvfp4 | lim(fp16) | lim(nvfp4)
  16 |     30.1 |     60.2 |     107.1 |       HBM |        HBM
  32 |     60.0 |    120.0 |     213.3 |       HBM |        MMA
  64 |    119.1 |    238.3 |     423.6 |       HBM |        MMA
 128 |    234.8 |    469.7 |     835.0 |       MMA |        MMA

=== NVIDIA B300 (Blackwell Ultra, GB300) (sm_103) | HBM 8000.0 GB/s | FP16-TC ridge 312.5 | GQA-8 fp16 AI 8.0 ===
 h_q |  AI fp16 |   AI fp8 |  AI nvfp4 | lim(fp16) | lim(nvfp4)
  16 |     30.1 |     60.2 |     107.1 |       HBM |        HBM
  32 |     60.0 |    120.0 |     213.3 |       HBM |        HBM
  64 |    119.1 |    238.3 |     423.6 |       HBM |        MMA
 128 |    234.8 |    469.7 |     835.0 |       HBM |        MMA

PURE ROOFLINE: at h_q=128, fp16 sits ~0.75x the FP16-TC ridge; fp8/nvfp4 latent push AI PAST it
-> a limiter FLIP to compute (first in the v1->v11 arc). PER-CT

## 3. Build v11_mla (JIT) — single transposed FP16 latent tile + fused NVFP4 dequant

In [4]:
import glob, os, shutil
for d in glob.glob(os.path.expanduser('~/.cache/torch_extensions/*/fa_v11_mla')):
    if not glob.glob(os.path.join(d, '*.so')):
        shutil.rmtree(d, ignore_errors=True); print('cleaned stale build:', d)
from bindings.load import build_kernel
# CUDA-core / dequant-to-FP16. The (576,512) real-DeepSeek template stages ~46 KB of smem (sK_T 38 KB +
# sQ 9 KB) -- just under the T4's 48 KB static limit (the per-CTA / smem-capacity story, kickoff §9 Q2).
# If ptxas complains about smem, that IS the Q2 finding (drop to the 96/64 smoke shape or opt-in dynamic smem).
mla = build_kernel('v11_mla'); print('built v11 (MLA latent-KV):', mla)

built v11 (MLA latent-KV): <module 'fa_v11_mla' from '/root/.cache/torch_extensions/py312_cu128/fa_v11_mla/fa_v11_mla.so'>


## 4. Correctness gate — v11_mla + v10/v8.7 regression (Gate 1 of 2)

Decode `h_q` sweep {16,32,64,128} x non-multiple N_k x causal x seed; idle-warp (h_q=3) + multi-row-tile (h_q=20); the real (576,512) latent; the square-prefill reduction; AND the **absorption identity** (the latent-absorbed kernel == explicit per-head materialization, kickoff §9 Q4).

In [5]:
!python -m pytest tests/test_correctness.py -k "v11_mla or v10_nvfp4 or v8_gqa_ss" -q

........................................................................ [ 48%]
........................................................................ [ 97%]
....                                                                     [100%]
148 passed, 354 deselected in 167.04s (0:02:47)


## 5. Accuracy — NVFP4-latent storage cost vs FP16 latent (and FP8 latent for comparison)

Three numbers per shape: (1) kernel vs the apples-to-apples oracle (MQA over the SAME dequantized NVFP4 latent) -> FP16 band; (2) NVFP4-latent quant RMSE vs the raw fp16 latent (the real number); (3) FP8-latent quant RMSE for comparison.

In [6]:
import torch
from fa_kernels import mla_attention
from fa_kernels.paged import (build_paged_kv_mla, quantize_nvfp4, dequantize_nvfp4,
                              quantize_fp8_e4m3, dequantize_fp8_e4m3)
from fa_kernels.reference import sdpa_reference_mla

def rmse(a, b):
    return (a - b).pow(2).mean().sqrt().item()

def mqa_latent(q, lat, DV, scale):                      # MQA over a (dense fp16) latent, latent-basis
    sc = torch.matmul(q, lat.transpose(-1, -2)) * scale
    p = torch.softmax(sc - sc.amax(dim=-1, keepdim=True), dim=-1)
    return torch.matmul(p, lat[..., :DV])

print(f"{'shape(h_q/N_k DQK)':>20} | {'vs MLA oracle':>13} | {'NVFP4 vs fp16':>14} | {'FP8 vs fp16':>12} | verdict")
for (L, R) in ((64, 32), (512, 64)):                    # the smoke shape and the real DeepSeek-V3 shape
    DQK = L + R
    for h_q in (8, 128):
        for seed in (9, 17):
            torch.manual_seed(seed)
            B, N_k = 1, 8192
            q = torch.randn(B, h_q, 1, DQK, device='cuda')
            latent = torch.randn(B, 1, N_k, DQK, device='cuda')
            scale = 1.0 / (DQK ** 0.5)
            lp, lm, bt, nk, sl = build_paged_kv_mla(latent, 256, seed=seed)
            out = mla_attention(q, lp, lm, bt, 256, nk, L, sl, causal=False, q_offset=0)
            # (1) apples-to-apples: oracle dequantizes the SAME NVFP4 latent bytes.
            r_or = rmse(out, sdpa_reference_mla(q, latent, L, causal=False))
            # (2) NVFP4-latent quant RMSE: vs MQA over the raw fp16 latent (no quant).
            r4 = rmse(out, mqa_latent(q, latent.to(q.dtype), L, scale))
            # (3) FP8-latent quant RMSE for comparison (dequant the SAME fp16 latent through E4M3).
            lb, sl8 = quantize_fp8_e4m3(latent); l8 = dequantize_fp8_e4m3(lb, sl8).to(latent.dtype)
            r8 = rmse(mqa_latent(q, l8, L, scale), mqa_latent(q, latent.to(q.dtype), L, scale))
            verdict = 'FP4~FP8' if r4 < 2 * r8 else 'FP8 floor?'
            print(f"{f'{h_q}/{N_k} D{DQK}/{L} s{seed}':>20} | {r_or:13.2e} | {r4:14.2e} | {r8:12.2e} | {verdict}")
print('\n(1) sits in the FP16 band (kernel math correct). (2) is the NVFP4-latent quant RMSE (the deliverable).')
print('If NVFP4 >> FP8, the honest result is "FP8 is the accuracy floor; FP4 latent buys capacity at cost X".')

  shape(h_q/N_k DQK) | vs MLA oracle |  NVFP4 vs fp16 |  FP8 vs fp16 | verdict
    8/8192 D96/64 s9 |      4.86e-05 |       2.66e-03 |     8.45e-04 | FP8 floor?
   8/8192 D96/64 s17 |      2.18e-05 |       2.32e-03 |     6.54e-04 | FP8 floor?
  128/8192 D96/64 s9 |      4.31e-05 |       2.58e-03 |     7.73e-04 | FP8 floor?
 128/8192 D96/64 s17 |      2.44e-05 |       2.59e-03 |     7.63e-04 | FP8 floor?
  8/8192 D576/512 s9 |      1.17e-05 |       2.50e-03 |     6.96e-04 | FP8 floor?
 8/8192 D576/512 s17 |      1.11e-05 |       2.40e-03 |     6.97e-04 | FP8 floor?
128/8192 D576/512 s9 |      1.17e-05 |       2.48e-03 |     6.96e-04 | FP8 floor?
128/8192 D576/512 s17 |      1.10e-05 |       2.46e-03 |     6.98e-04 | FP8 floor?

(1) sits in the FP16 band (kernel math correct). (2) is the NVFP4-latent quant RMSE (the deliverable).
If NVFP4 >> FP8, the honest result is "FP8 is the accuracy floor; FP4 latent buys capacity at cost X".


## 6. Capacity — MLA's headline (the ~93% KV reduction), MEASURED

MLA stores ONE latent (`DQK = L+R` dims/token) vs MHA's `2*H*d` and GQA's `2*H_kv*d`. NVFP4 packs the latent at 0.5625 B/elem on top. Report the per-token byte footprint.

In [7]:
import torch
from fa_kernels.paged import build_paged_kv, build_paged_kv_mla
B, N_k, ps = 1, 65536, 256
H, d = 128, 128          # a DeepSeek-class MHA: 128 heads x 128 dims
H_kv = 8                 # a GQA-8 comparator
L, R = 512, 64; DQK = L + R
def pool_bytes(*ts): return sum(t.numel() * t.element_size() for t in ts)
# MHA / GQA full KV (fp16), vs the MLA latent (NVFP4, one head).
k_mha = torch.randn(B, H,    N_k, d, device='cuda', dtype=torch.float16)
v_mha = torch.randn(B, H,    N_k, d, device='cuda', dtype=torch.float16)
k_gqa = torch.randn(B, H_kv, N_k, d, device='cuda', dtype=torch.float16)
v_gqa = torch.randn(B, H_kv, N_k, d, device='cuda', dtype=torch.float16)
latent = torch.randn(B, 1, N_k, DQK, device='cuda', dtype=torch.float16)
b_mha = pool_bytes(*build_paged_kv(k_mha, v_mha, ps)[:2])
b_gqa = pool_bytes(*build_paged_kv(k_gqa, v_gqa, ps)[:2])
lp, lm, _, _, _ = build_paged_kv_mla(latent, ps)
b_mla = pool_bytes(lp, lm)                               # packed nibbles + micro-scales (count BOTH)
print(f'KV-cache footprint for N_k={N_k} (page_size {ps}):')
print(f'  MHA  fp16 (H={H},d={d})   : {b_mha/1e6:8.2f} MB  ({2*H*d} dims/token)')
print(f'  GQA-8 fp16 (H_kv={H_kv},d={d}) : {b_gqa/1e6:8.2f} MB  ({2*H_kv*d} dims/token)  -> {b_mha/b_gqa:.1f}x smaller than MHA')
print(f'  MLA  NVFP4 (latent {DQK})   : {b_mla/1e6:8.2f} MB  ({DQK} dims/token)  -> {b_mha/b_mla:.1f}x smaller than MHA, {b_gqa/b_mla:.1f}x vs GQA-8')
print(f'\nMLA latent is {100*(1-b_mla/b_mha):.1f}% smaller than MHA fp16 KV (shape reduction x NVFP4 bytes).')

KV-cache footprint for N_k=65536 (page_size 256):
  MHA  fp16 (H=128,d=128)   :  4294.97 MB  (32768 dims/token)
  GQA-8 fp16 (H_kv=8,d=128) :   268.44 MB  (2048 dims/token)  -> 16.0x smaller than MHA
  MLA  NVFP4 (latent 576)   :    21.23 MB  (576 dims/token)  -> 202.3x smaller than MHA, 12.6x vs GQA-8

MLA latent is 99.5% smaller than MHA fp16 KV (shape reduction x NVFP4 bytes).


## 7. THE T1 A/B — does MLA leave the per-CTA floor? (%HBM vs N_k at h_q=128)  ⚠️ T4 latency NOT valid

The decode columns: us/tok (NOT valid on T4 — emulated FP4 ALU), **%HBM** (the per-CTA-floor signal), and the roofline limiter. The T1 question: does MLA's %HBM climb off the ~0.5%/per-CTA floor GQA decode never left? The honest read is the **%HBM trend**, not absolute latency. Real shape `--dim 576 --heads 128`.

In [8]:
# MLA decode at the real DeepSeek-V3 latent (576/512), h_q=128, across N_k. %HBM uses the ONE-latent
# byte count (read once for all heads). vs-naive is N/A for MLA (no clean same-shape no-packing baseline;
# the matched-work A/B vs v10-GQA is a modeling choice). Read %HBM + roofline, not us/tok (T4 emulated FP4).
print('=== v11 MLA (real DeepSeek-V3 latent 576/512), h_q=128, N_k sweep ===')
!python -m bench.harness --backend v11_mla --decode --seq 2048 8192 16384 --heads 128 --dim 576
print('\n=== v11 MLA (smoke latent 96/64), h_q sweep via repeated --heads ===')
for hq in (16, 32, 64, 128):
    !python -m bench.harness --backend v11_mla --decode --seq 8192 --heads {hq} --dim 96

=== v11 MLA (real DeepSeek-V3 latent 576/512), h_q=128, N_k sweep ===
# device: Tesla T4 (sm_75)  clock~585/1590MHz  backend=v11_mla  precision=fp32  causal=False  decode=True
#       shape(q x kv) |    ours p50/max ms |   us/tok |   %HBM |  vs sdpa | vs naive | roofline
ninja: no work to do.
     1x128x1x576/2048 |   2.341/  2.443 |    18.29 |   0.1% |    0.95x |     nanx | MMA (~0.01ms)
     1x128x1x576/8192 |   6.087/  6.392 |    47.55 |   0.1% |    2.00x |     nanx | MMA (~0.04ms)
    1x128x1x576/16384 |  12.071/ 12.193 |    94.30 |   0.1% |    2.03x |     nanx | MMA (~0.07ms)

=== v11 MLA (smoke latent 96/64), h_q sweep via repeated --heads ===
# device: Tesla T4 (sm_75)  clock~585/1590MHz  backend=v11_mla  precision=fp32  causal=False  decode=True
#       shape(q x kv) |    ours p50/max ms |   us/tok |   %HBM |  vs sdpa | vs naive | roofline
ninja: no work to do.
       1x16x1x96/8192 |   0.260/  0.350 |    16.27 |   0.5% |    1.38x |     nanx | HBM (~0.00ms)
# device: Tesla T4 (

## 8. [RENT root B300/sm_103] — the measured core (latency / regime crossover / native FP4)

**Gate 2 + the paper's record run, NOT on Colab T4.** Carry v10's honesty debts: **lock clocks** (root/bare-metal), **install + run ncu on a privileged box** (counter-validate the %HBM proxy), **regenerate nsys in-notebook with 2025.3.2+**. Deliverables:

1. **The crossover sweep** (`bench.regime`): %HBM (+ achieved TFLOPS) vs N_k past the ~126 MB B300 L2, clock-locked, L2-flushed. Does MLA's higher AI finally reach a bandwidth- or compute-bound regime, or stay per-CTA? `python -m bench.regime --backend v11_mla --dim 576 --h-kv 1 --gqa-group 128 --lock-clocks --kv-lens 8192 32768 131072 524288`
2. **Same-session clock-matched `vs naive`** (the only trustworthy latency number) vs v10-GQA at matched NVFP4 storage — isolates the SHAPE.
3. **sm_103 2x-exp re-ablation** now that M=128 puts many exps in flight (was a dud at M=1 in v10).
4. **Native FP4 tcgen05 arm** — ONLY-IF the dev rung proves M=128 packs as ONE GEMM (§9 Q1) and the limiter flips (§9 Q2).
5. **Comparators, clock-locked, matched precision: FlashMLA / FlashInfer trtllm-gen.**

In [9]:
# Template for the root-B300 regime sweep (the T1 crossover deliverable). Clock-lock needs root; on an
# unprivileged box the counter-free %HBM proxy still runs (eff_bw vs HBM peak), only cross-run wall-times
# are confounded. The L2-flush + the 1 GB+ working set at large N_k kill the L2-residency confound.
!python -m bench.regime --backend v11_mla --dim 576 --h-kv 1 --gqa-group 128 \
    --kv-lens 8192 32768 131072 524288 --lock-clocks
# Byte-only / shape comparators on the SAME box (run v10-GQA + v8.7 for the matched A/B):
# !python -m bench.regime --backend v10_nvfp4 --dim 128 --h-kv 1 --gqa-group 8 --kv-lens 8192 32768 131072 --lock-clocks

# clocks LOCKED: sm=1590MHz (target 1590) mem=5001MHz  (no throttle)
# device: Tesla T4 (sm_75)  clock~1590/1590MHz  backend=v11_mla  KV=nvfp4-latent  l2_flush=True  gqa_group=128
#   shape(BxHq x1x d /Nk) Hkv |   us/tok |    eff_bw |   %HBM |   WS(MB) | L2res? | L2served
ninja: no work to do.
        1x128x1x576/8192 Hkv1 |    47.56 |     0.4GB |   0.1% |     2.65 |    yes |        -
       1x128x1x576/32768 Hkv1 |   188.03 |     0.4GB |   0.1% |    10.62 |     no |        -
  # skip B1 H_kv1 d576 N131072: OutOfMemoryError CUDA out of memory. Tried to allocate 576.00 MiB. GPU 0 has 
  # skip B1 H_kv1 d576 N524288: OutOfMemoryError CUDA out of memory. Tried to allocate 1.12 GiB. GPU 0 has a 
# clocks reset.
